# Encontro 4 · Hands-on: a divisão se paga?

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-04/02-hands-on-multiagente.ipynb)

No notebook 01 vocês viram três arquiteturas por dentro. Aqui o grupo decide, com o próprio placar, qual delas defenderia para a Aurora, e por quê.

### O que o grupo entrega

`encontro-04/NOTA-ARQUITETURA.md` preenchida no fork da equipe. É uma página com quatro partes:

1. o **padrão** escolhido, nomeado: agente único, roteador + especialistas ou supervisor + especialistas;
2. o **diagrama**, partindo do modelo em Mermaid que já está na nota;
3. o **placar** que este notebook gera no Passo 4, pronto para colar;
4. a **decisão**: a divisão se paga? Defendida com um caso concreto do placar.

**Decidir não dividir é uma resposta tão válida quanto dividir.** O que vale nota é a decisão sustentada por números, e não a arquitetura mais sofisticada.

### Como trabalhar

| Papel | O que faz |
|---|---|
| **Quem roda** | Compartilha a tela e executa as células |
| **Quem escreve** | Preenche a nota enquanto o grupo discute |
| **Quem lê o trace** | No Passo 3, conduz o diagnóstico do caso escolhido |

Rodem as células de cima para baixo. Quando o professor perguntar "em que passo vocês estão?", respondam no chat com o número.

| Passo | Tempo sugerido |
|---|---|
| 1 · Linha de base | 5 min |
| 2 · Mudar uma coisa de cada vez | 20 min |
| 3 · Abrir um caso | 15 min |
| 4 · Escrever a nota | 10 min |

**Regra do dia:** durante a aula, só este notebook. Entregas atrasadas ficam para depois da aula, no prazo que o professor combinou.

**Nada aqui precisa de chave.** Todas as opções do menu estão gravadas (veja "Por que não precisa de chave de API" no notebook 01).

In [1]:
# Etapa 0 · No Colab, baixa o repositório da disciplina. Na sua máquina, não faz nada.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-04")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"], check=True)
sys.path.insert(0, os.getcwd())

import multiagente as ma
ma.preparar("usar")
casos = ma.carregar_casos()
print(f"{len(casos)} casos. Tudo certo, pode seguir.")
experimentos = []

Replay 'usar': 333 respostas gravadas com gpt-4o-mini em 2026-10-12 15:36
Há chave configurada: o que não estiver gravado será chamado ao vivo, com custo.
Busca: 90 trechos da wiki da Aurora, espaços confluence-rh, wiki-beneficios, wiki-financeiro, wiki-rh, wiki-ti
19 casos. Tudo certo, pode seguir.


## Passo 1 · A linha de base

**Produz:** a primeira linha do placar, com o agente único da Aula 3.

Toda decisão de arquitetura é uma comparação com alguma coisa. Aqui, a comparação é sempre com o agente único: um agente, uma busca na wiki inteira. Se uma arquitetura mais complexa não superar essa linha em algum critério que importa, ela não se paga.

In [2]:
base = ma.rodar(ma.monolito(), casos, mostrar=False)
ma.registrar(experimentos, base)
ma.placar(*experimentos)

arquitetura            acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
agente único (Aula 3)  16/19    17/17       2.6            3007         0.0655             0.00409             2.7     


### Lembrete: como ler o placar

| Coluna | O que olhar |
|---|---|
| acertos | quantos dos 19 casos saíram certos (desfecho, fontes e valor) |
| rota certa | se os agentes acionados enxergavam a parte da wiki certa. É aqui que um roteador ruim aparece |
| chamadas/caso | o preço da coordenação, em chamadas ao modelo |
| tokens/caso | o preço da coordenação, em tokens |
| custo/acerto (US$) | **o número que decide:** quanto custa uma resposta certa |
| seg/caso | o tempo que o colaborador espera |

## Passo 2 · Mudar uma coisa de cada vez

**Produz:** uma linha nova no placar a cada vez que vocês rodam a célula.

### Por que uma coisa de cada vez

Se vocês mudarem duas opções e o acerto subir, não dá para saber qual das duas fez a diferença. É o mesmo princípio do hands-on do Encontro 3: uma mudança por rodada, e o placar explica o que aconteceu.

### O menu

Cada opção testa uma hipótese de arquitetura:

| Opção | Valores | A pergunta que ela testa | Onde olhar |
|---|---|---|---|
| `ROTEADOR` | `"llm"` · `"regra"` | Vale pagar uma chamada ao modelo para decidir a rota, ou palavras-chave bastam? | rota certa, custo/acerto |
| `EQUIPE` | `3` · `4` | O Financeiro merece um especialista próprio? Com 4 opções, o roteador tem mais por onde errar, mas cada descrição fica mais estreita | rota certa nos casos E1, E2, R3 e R1 |
| `HANDOFF` | `"pergunta_original"` · `"resumo_do_roteador"` | Reescrever a pergunta ajuda o especialista, ou tira dele o detalhe que decidia? (O resumo só existe com o roteador `"llm"`) | acertos, e o texto passado adiante no trace |
| `SE_INCERTO` | `"seguir"` · `"agente_unico"` | Quando o roteador não tem certeza, é melhor arriscar a rota ou devolver a pergunta ao agente único? | acertos, rota certa |

**Um alerta sobre `SE_INCERTO = "agente_unico"`.** Devolver ao agente único é uma forma de degradação graciosa: quando o roteador não sabe, quem enxerga tudo responde. Mas, nesse caminho, **a parede some**, porque o agente único enxerga todos os espaços da wiki. Se o controle de acesso por fonte for uma exigência, e no caso da Aurora é, esse fallback viola a exigência justamente nos casos ambíguos. O placar não mostra isso: é o grupo que precisa enxergar.

### Um roteiro possível

1. Rodem a célula como está: roteador por modelo, 3 especialistas, pergunta original, seguir.
2. Troquem **só** o `ROTEADOR` para `"regra"` e rodem.
3. Voltem para `"llm"` e troquem **só** a `EQUIPE` para `4`.
4. Testem o `HANDOFF`, depois o `SE_INCERTO`.
5. Ponham o supervisor no placar, na célula seguinte.

Cada rodada recebe um nome automático, com a configuração usada. Rodar a mesma configuração de novo substitui a linha, sem duplicar.

In [7]:
ROTEADOR   = "regra"                  # "llm" ou "regra"
EQUIPE     = 3                      # 3 ou 4
HANDOFF    = "pergunta_original"    # "pergunta_original" ou "resumo_do_roteador"
SE_INCERTO = "seguir"               # "seguir" ou "agente_unico"

rodada = ma.rodar(ma.roteador_especialistas(ROTEADOR, EQUIPE, HANDOFF, SE_INCERTO), casos, mostrar=False)
ma.registrar(experimentos, rodada)
ma.placar(*experimentos)

arquitetura                                        acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
agente único (Aula 3)                              16/19    17/17       2.6            3007         0.0655             0.00409             2.7     
roteador llm · 3 esp · original · incerto→segue    16/19    14/17       3.1            2588         0.0599             0.00375             3.2     
supervisor · 3 esp                                 17/19    15/17       3.9            3370         0.079              0.00465             4.2     
roteador regra · 3 esp · original · incerto→segue  16/19    14/17       2.0            2279         0.0506             0.00316             2.1     


### O supervisor

O supervisor consulta quantos especialistas quiser e junta as respostas. Resolve as perguntas de duas áreas, mas relê tudo a cada passo. A célula abaixo põe no placar o supervisor com a `EQUIPE` escolhida acima:

In [5]:
ma.registrar(experimentos, ma.rodar(ma.supervisor(EQUIPE), casos, mostrar=False))
ma.placar(*experimentos)

arquitetura                                      acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
agente único (Aula 3)                            16/19    17/17       2.6            3007         0.0655             0.00409             2.7     
roteador llm · 3 esp · original · incerto→segue  16/19    14/17       3.1            2588         0.0599             0.00375             3.2     
supervisor · 3 esp                               17/19    15/17       3.9            3370         0.079              0.00465             4.2     


**Pergunta para o grupo.** Antes de seguir, escrevam em uma frase o que mais mexeu no placar: o roteador, o tamanho da equipe, o handoff ou a regra para quando há incerteza. Mexeu no acerto ou só no custo?

> **Para ir além (opcional).** O roteador por regra não gasta nenhum token. Em quantos casos ele erra a rota que o modelo acerta? Esse erro se paga com a economia? Em que tipo de empresa a resposta mudaria?

In [ ]:
# # Travou, ou o placar ficou bagunçado? Descomente e rode: refaz um placar de referência.
# experimentos = []
# for arq in [ma.monolito(), ma.roteador_especialistas("llm", 3), ma.roteador_especialistas("regra", 3),
#             ma.supervisor(3)]:
#     ma.registrar(experimentos, ma.rodar(arq, casos, mostrar=False))
# ma.placar(*experimentos)

arquitetura                                        acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
agente único (Aula 3)                              16/19    17/17       2.6            3007         0.0655             0.00409             2.7     
roteador llm · 3 esp · original · incerto→segue    16/19    14/17       3.1            2588         0.0599             0.00375             3.2     
roteador regra · 3 esp · original · incerto→segue  16/19    14/17       2.0            2279         0.0506             0.00316             2.1     
supervisor · 3 esp                                 17/19    15/17       3.9            3370         0.079              0.00465             4.2     


## Passo 3 · Abrir um caso

**Produz:** o diagnóstico de um caso em que a arquitetura do grupo se saiu diferente da linha de base.

### Por que abrir um caso

O placar diz **quanto** mudou. O trace diz **por quê**. Uma nota que só repete números ("acertou 2 a mais") não convence. Uma nota que mostra o mecanismo ("o roteador mandou o C2 para Benefícios porque a descrição de Benefícios fala em auxílios") convence, e ainda diz o que consertar.

Primeiro, os casos em que a última arquitetura testada e a linha de base discordam:

In [8]:
ma.onde_mudou(experimentos[0], experimentos[-1])

caso  agente único (Aula 3)               roteador regra · 3 esp · original 
T2    ok                                  ERR desfecho 'escalar', esperado '
S1    ERR desfecho 'escalar', esperado '  ok
S2    ERR desfecho 'escalar', esperado '  ok
R1    ok                                  ERR não citou ti-vpn-e-acesso-remo


Escolham um desses casos e abram o trace. Prefiram um caso que a arquitetura do grupo **errou**: é o que mais ensina. Se nenhum caso mudou, escolham o R1, que é o mais rico.

Se precisarem relembrar como ler o trace, a tabela está na Etapa 4 do notebook 01.

In [9]:
CASO = "R1"          # troque por um dos ids acima
ma.mostrar_trace(experimentos[-1], CASO)

R1 · Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
arquitetura: roteador regra · 3 esp · original · incerto→segue

  [roteador] regra: palavras: vpn, wi-fi, viajar, hotel, alimentacao
  [roteador] → rh (confiança alta)  · palavras: ['vpn', 'wi-fi', 'viajar', 'hotel', 'alimentacao']
  [roteador] passa adiante → especialista rh: “Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?”
      [especialista rh] acessa: wiki-rh, confluence-rh, wiki-financeiro
      [especialista rh] recebeu: Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
      [especialista rh] modelo pediu: buscar_politicas_rh, buscar_politicas_rh   (685 tokens de entrada, 69 de saída)
      [especialista rh] buscar_politicas_rh({"pergu

### Classifiquem a causa

| Causa | Como reconhecer no trace | O que consertaria |
|---|---|---|
| **Rota errada** | O roteador escolheu um especialista que não enxerga o espaço da resposta | A descrição dos especialistas, ou a lista de palavras da regra |
| **Handoff perdeu informação** | O texto passado adiante não tem o detalhe que decidia a resposta | Passar a pergunta original, ou proibir certos cortes no resumo |
| **Pergunta de duas áreas** | Um especialista só não cobre tudo o que foi perguntado | Um supervisor, ou dividir a pergunta antes de rotear |
| **Parede funcionando** | O especialista disse "não sei" porque a fonte é de outra área, e isso é o desejado | Nada: é a arquitetura protegendo uma fonte. Mas a rota estava certa? |
| **Só custo** | O desfecho não mudou; só cresceram as chamadas e os tokens | É o argumento para **não** dividir |

Ponham na nota o id do caso, a causa e a linha do trace que a mostra.

In [ ]:
# Para ver também o prompt de sistema e as ferramentas de cada agente:
# ma.mostrar_trace(experimentos[-1], CASO, completo=True)

R1 · Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
arquitetura: roteador regra · 3 esp · original · incerto→segue

  [roteador] regra: palavras: vpn, wi-fi, viajar, hotel, alimentacao
  [roteador] → rh (confiança alta)  · palavras: ['vpn', 'wi-fi', 'viajar', 'hotel', 'alimentacao']
  [roteador] passa adiante → especialista rh: “Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?”
      [especialista rh] acessa: wiki-rh, confluence-rh, wiki-financeiro
      [especialista rh] recebeu: Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
         ferramentas: buscar_politicas_rh, responder, escalar_para_rh
         | Você responde dúvidas internas de colaboradores da Aurora Tecnologia.
         | 
      

## Passo 4 · Escrever a nota

**Produz:** o placar em markdown e a configuração escolhida, prontos para colar em `NOTA-ARQUITETURA.md`.

Na célula abaixo, troquem o índice de `escolhida` pelo da rodada que o grupo defende. O placar está na ordem em que as rodadas foram feitas: `0` é a linha de base, `1` é a segunda rodada, e `-1` é a última.

In [11]:
print(ma.tabela_markdown(experimentos))
print()
escolhida = experimentos[-1]          # troque pelo índice da rodada que o grupo defende
print("Configuração defendida:", escolhida["nome"])
print(escolhida["config"])

| arquitetura | acertos | rota certa | chamadas/caso | tokens/caso | custo total (US$) | custo/acerto (US$) | seg/caso |
|---|---|---|---|---|---|---|---|
| agente único (Aula 3) | 16/19 | 17/17 | 2.6 | 3007 | 0.0655 | 0.00409 | 2.7 |
| roteador llm · 3 esp · original · incerto→segue | 16/19 | 14/17 | 3.1 | 2588 | 0.0599 | 0.00375 | 3.2 |
| supervisor · 3 esp | 17/19 | 15/17 | 3.9 | 3370 | 0.079 | 0.00465 | 4.2 |
| roteador regra · 3 esp · original · incerto→segue | 16/19 | 14/17 | 2.0 | 2279 | 0.0506 | 0.00316 | 2.1 |

Configuração defendida: roteador regra · 3 esp · original · incerto→segue
{'arquitetura': 'roteador + especialistas', 'roteador': 'regra', 'equipe': 3, 'handoff': 'pergunta_original', 'se_incerto': 'seguir'}


### Como escrever a decisão

A seção 4 da nota é a mais importante. Ela responde "a divisão se paga?" com números do placar e com o motivo de arquitetura. Compare dois jeitos de escrever:

> **Fraca:** "Escolhemos o supervisor porque é mais robusto e escalável."

Não tem número, não diz em relação a quê e usa adjetivos que qualquer arquitetura poderia reivindicar.

> **Forte:** "Escolhemos [padrão e configuração]. Ele acertou [N]/19, contra [M]/19 do agente único, e custou [X] vezes mais por acerto. O ganho veio de [casos], todos de [causa]. Mantemos a divisão porque [a parede de acesso por fonte é exigência / o ganho em X paga o custo]. Se [condição], voltaríamos ao agente único."

Diz o que foi escolhido, contra o quê, com que números, de onde veio a diferença e o que faria a decisão mudar. A última frase é o critério de reversão, o mesmo do ADR do Encontro 1.

Se o grupo decidiu **não** dividir, a estrutura é a mesma: "o agente único acertou tanto quanto [arquitetura] e custou [X] vezes menos por acerto; a divisão só se pagaria se [condição]".

### Confiram antes de entregar

- [ ] O padrão está nomeado, e o diagrama mostra quem chama quem e que espaços cada agente enxerga.
- [ ] A tabela tem a linha de base e pelo menos duas configurações.
- [ ] A decisão tem números, cita um caso pelo id e diz a causa (Passo 3).
- [ ] Há um critério de reversão: o que faria o grupo mudar de ideia.

## Para ir além (opcional, com chave)

Tudo acima está gravado. Mudar um texto que o modelo lê, como a descrição de um especialista, gera chamadas que não estão na gravação. Para elas, é preciso uma chave de API, e cada rodada nova tem custo, em geral de centavos com um modelo barato.

Só façam isto se o professor liberou uma chave para o grupo. No Colab, guardem a chave em **Segredos**, o ícone de chave na barra lateral, com o nome `API_KEY`. Nunca colem a chave numa célula: o notebook pode ser compartilhado com ela dentro.

O que já está gravado continua vindo da gravação, sem custo; só o que for novo vai para a API. Para desfazer uma mudança nas descrições, use **Ambiente de execução → Reiniciar sessão** e rode o notebook de novo.

In [ ]:
# [não gravar] Só com chave. O que já está gravado continua vindo da gravação, sem custo.
# from google.colab import userdata
# ma.usar_chave(userdata.get("API_KEY"))
#
# ma.AREAS["rh"]["descricao"] += ", inclusive o auxílio home office"
# nova = ma.rodar(ma.roteador_especialistas("llm", 3), casos, nome="rh descreve o auxílio home office")
# ma.registrar(experimentos, nova)
# ma.placar(*experimentos)

ModuleNotFoundError: No module named 'google.colab'